# 08 · S3 store and AWS credentials

This notebook exercises `dtrack_left.backends.store` (`S3Location`, `S3Store`, `fetch_report`) on a real S3 scratch prefix, and `dtrack_left.backends.aws` (`session`, `resolve`, `whoami`, `refresh_profile`, `write_profile`, `next_delay`, `watch`, `winscp_path`) against your real AWS profile.

It mirrors `tests/unit/dtrack_left/backends/test_store.py` and `tests/unit/dtrack_left/backends/test_aws.py`, which use moto and a fake `credential_process`. Here every call goes to real S3/STS; S3 writes stay under `S3_SCRATCH`, credential files are written into WORKDIR, and the real `~/.aws/credentials` is only touched when `DRY_RUN = False`.

## How to approach

**Prerequisites**
- Kernel: the repo `.venv` (`uv sync`; boto3 is in the base install).
- An `~/.aws/config` profile (`DTRACK_AWS_PROFILE`, default `corp`) that works with `aws sts get-caller-identity --profile corp`, usually via `credential_process` or SSO.
- `DTRACK_PROD_S3` set to a scratch prefix you may write to, for example `s3://<bucket>/dtrack/nb-scratch`. The notebook adds a per-run folder under it and deletes it at the end.
- `DTRACK_PROD_CONFIG` / `DTRACK_NB_PAIR`: only used to put real pair metadata into the store (no database connection).

**Run order**: top to bottom. Sections 1-7 are S3 (fast, a few dozen small objects). Sections 8-11 are credentials: they mint real temporary keys and write them only to `WORKDIR/credentials`. Section 12 is the real `~/.aws/credentials` write, gated by `DRY_RUN`. Always run Cleanup.

**What to look at**: every key lives under the run prefix; `list_keys` returns relative names; `ready_unclaimed` goes 2 -> 0; the static profile written to WORKDIR authenticates as the same ARN as the source profile.

**Failure modes**
- `AccessDenied` on put/list - the profile cannot write the scratch prefix; pick another `S3_SCRATCH`.
- `no AWS credentials for profile` - the profile exists but has no keys/credential_process/SSO session (run `aws sso login`).
- `NoRegionError` - set `region` in the profile or `AWS_DEFAULT_REGION`.
- `exists()` raising instead of returning False - S3 returned 403, not 404 (no `s3:ListBucket` on the bucket).

In [ ]:
import os
from pathlib import Path

CONFIG_PATH = os.environ.get("DTRACK_PROD_CONFIG", "C:/work/dtrack/c.json")   # your real pairs config
PAIR = os.environ.get("DTRACK_NB_PAIR", "orders")                             # pair whose metadata goes into the store
ENV_FILE = os.environ.get("DTRACK_ENV_FILE", "")                              # "" = search for .env upward from the cwd
S3_SCRATCH = os.environ.get("DTRACK_PROD_S3", "s3://<bucket>/dtrack/nb-scratch")  # writable scratch prefix
AWS_PROFILE = os.environ.get("DTRACK_AWS_PROFILE", "corp")                    # ~/.aws/config profile to test
AWS_TARGET = os.environ.get("DTRACK_AWS_TARGET", "corp-winscp")               # static profile name refresh_profile writes
S3_WITH_PROFILE = True                                                        # True = S3Store uses AWS_PROFILE; False = default boto3 chain (what the CLI does)
REPORT_S3 = os.environ.get("DTRACK_REPORT_S3", "")                            # optional real report store to fetch from (read-only); "" = demo under scratch
FORBIDDEN_S3 = ""                                                             # optional s3://bucket/prefix you cannot read, to see exists() raise
PAGE_SIZE = 2                                                                 # MaxKeys used to force list pagination
WATCH_SECONDS = 20                                                            # how long the watch() demo runs (0 = skip)
WINSCP_SITE = "corp-s3"                                                       # saved WinSCP site name
LAUNCH_WINSCP = False                                                         # True = actually start WinSCP
WORKDIR = Path.home() / ".local/plans/nbs/_work/08"                           # local output folder
KEEP = False                                                                  # True = keep the S3 run prefix and WORKDIR
DRY_RUN = True                                                                # False = write AWS_TARGET into the real ~/.aws/credentials

In [ ]:
import json
import re
import shutil
from datetime import date, datetime, timedelta

import boto3
import pandas as pd
from dotenv import find_dotenv, load_dotenv

from dtrack_left.backends import aws
from dtrack_left.backends.store import S3Location, S3Store, fetch_report
from dtrack_left.cli.config import load_config

# project .env first (it may set AWS_PROFILE, DTRACK_S3_ENDPOINT_URL, ...)
env_file = ENV_FILE or find_dotenv(usecwd=True)
print("env file:", env_file or "(none found)")
if env_file:
    load_dotenv(env_file, override=False)

# real pair metadata to store; no database connection is made
config = load_config(CONFIG_PATH)
pair = config["pairs"][PAIR]
TO = (date.today() - timedelta(days=1)).isoformat()

# a fresh folder under the scratch prefix for this run
RUN_TAG = datetime.now().strftime("nb08-%Y%m%d-%H%M%S")
ROOT = f"{S3_SCRATCH.rstrip('/')}/{RUN_TAG}"
WORKDIR.mkdir(parents=True, exist_ok=True)
print("run prefix:", ROOT)


# >>> hide everything but the edges of a key or token <<< #
def redact(text):
    return re.sub(r"(=\s*)(\S{4})\S+(\S{4})", r"\1\2...\3", text)

## 1. Parse the location (`S3Location`)

`S3Location.parse` splits `s3://bucket/prefix` and strips slashes; anything that is not `s3://` raises `ValueError`.

In [ ]:
# the scratch root, the bare bucket, and a non-S3 path
print(S3Location.parse(ROOT))
print(S3Location.parse(f"s3://{S3Location.parse(ROOT).bucket}"))
try:
    S3Location.parse("/Volumes/main/dtrack/runs")
except ValueError as exc:
    print("ValueError:", exc)

## 2. Who am I (`whoami`, `resolve`)

`whoami(profile)` calls STS `GetCallerIdentity` and reports the account, ARN, region and how long the credentials last. `resolve(session)` returns the frozen keys and their expiry (None for static keys).

In [ ]:
# identity and expiry of the profile
identity = aws.whoami(AWS_PROFILE)
identity

In [ ]:
# how botocore found the keys, and when they expire
sess = aws.session(AWS_PROFILE)
frozen, expires = aws.resolve(sess)
print("method    :", sess.get_credentials().method)
print("access key:", f"{frozen.access_key[:4]}...{frozen.access_key[-4:]}")
print("has token :", bool(frozen.token))
print("expiry    :", aws._expiry(expires))

## 3. Open the store

`S3Store(root_uri)` builds its own boto3 client (honouring `DTRACK_S3_ENDPOINT_URL`); passing `client=` lets you pin it to a profile. The override settings are printed so you can see which endpoint is in use.

In [ ]:
# the S3 client: from the tested profile, or the default chain like the CLI
client = aws.session(AWS_PROFILE).client("s3") if S3_WITH_PROFILE else None
store = S3Store(ROOT, client=client)
print("bucket          :", store.bucket)
print("prefix          :", store.location.prefix)
print("endpoint        :", store.client.meta.endpoint_url)
print("addressing      :", (store.client.meta.config.s3 or {}).get("addressing_style", "auto"))
print("DTRACK_S3_ENDPOINT_URL:", os.getenv("DTRACK_S3_ENDPOINT_URL") or "(unset)")

## 4. The store contract

The same sequence the unit tests run on moto: `put_json`, `put_bytes`, `get_json`, `get_bytes`, `exists`, `list_keys` (with `start_after`). The payloads are your real pair metadata.

In [ ]:
# real pair metadata as JSON, and its col_map as a small CSV
meta = {
    "pair": PAIR,
    "left_source": pair["left"].get("source"),
    "right_kind": pair["right"].get("kind") or "table",
    "date_cols": [pair["left"]["date_col"], pair["right"]["date_col"]],
    "col_map_size": len(pair["col_map"]),
}
csv_body = ("left,right\n" + "".join(f"{k},{v}\n" for k, v in pair["col_map"].items())).encode()
print("put_json  ->", store.put_json("a/b.json", meta))
print("put_bytes ->", store.put_bytes("a/c.csv", csv_body, content_type="text/csv"))

# read back and compare
print("get_json equal :", store.get_json("a/b.json") == meta)
print("get_bytes equal:", store.get_bytes("a/c.csv") == csv_body)
print("exists b.json  :", store.exists("a/b.json"), "| exists missing:", store.exists("a/missing.json"))
print("list a/        :", store.list_keys("a/"))
print("after b.json   :", store.list_keys("a/", start_after="a/b.json"))
print("list nothing/  :", store.list_keys("nothing/"))

In [ ]:
# the raw S3 keys all sit under the run prefix; the content type travelled with the object
raw = store.client.list_objects_v2(Bucket=store.bucket, Prefix=store.location.prefix + "/")
print([row["Key"] for row in raw.get("Contents") or []])
head = store.client.head_object(Bucket=store.bucket, Key=store._key("a/b.json"))
print("ContentType:", head["ContentType"], "| SSE:", head.get("ServerSideEncryption"))

## 5. Pagination (`list_keys` follows continuation tokens)

S3 returns at most 1000 keys per page. To see the paging without writing thousands of objects, the real client is wrapped so each `list_objects_v2` call asks for only `PAGE_SIZE` keys. The calls are recorded: `StartAfter` must appear only on the first one.

In [ ]:
# a thin pass-through over the real S3 client that forces small pages and records the calls
class SmallPages:
    def __init__(self, client, size):
        self.client, self.size, self.calls = client, size, []

    def list_objects_v2(self, **kwargs):
        self.calls.append(dict(kwargs))
        return self.client.list_objects_v2(MaxKeys=self.size, **kwargs)

    def __getattr__(self, name):
        return getattr(self.client, name)


# five index-like keys, listed after the first one
for n in range(1, 6):
    store.put_json(f"paged/{n}.json", {"n": n, "pair": PAIR})
paged = S3Store(ROOT, client=SmallPages(store.client, PAGE_SIZE))
print(paged.list_keys("paged/", start_after="paged/1.json"))
for call in paged.client.calls:
    print({key: value for key, value in call.items() if key != "Bucket"})

## 6. Units, results, heartbeat, closing markers

These are the queue operations the distributed run uses: `commit_unit` writes the payload then an `_INDEX` pointer, `ready_unclaimed` counts index entries without a result or dead letter, `heartbeat`/`heartbeat_age` track liveness, and `close_*` write the closing markers.

In [ ]:
# two units committed, then one finished and one dead
first = store.commit_unit(1, "u1", {"window": [TO], "pair": PAIR}, {"pair": PAIR})
store.commit_unit(2, "u2", {"window": [TO], "pair": PAIR}, {"pair": PAIR})
print("pointer       :", first)
print("index         :", [key for key, _ in store.list_json("_INDEX/")])
print("ready before  :", store.ready_unclaimed())
store.put_result("u1", {"verdict": "match"})
store.put_dead("u2", {"error": "demo"})
print("ready after   :", store.ready_unclaimed())

# heartbeat: none yet, then a fixed time, then real age
print("age (none)    :", store.heartbeat_age())
store.heartbeat(now=100.0)
print("age @130 / @50:", store.heartbeat_age(now=130.0), store.heartbeat_age(now=50.0))
store.heartbeat()
print("age now       :", store.heartbeat_age())

# closing markers sort their pair names
store.close_row_check([PAIR, "zz_other"])
store.close_manifests([PAIR, "zz_other"], 4)
print(store.get_json("row_check/_CLOSED"), store.get_json("manifest/_CLOSED"))

## 7. `exists()` errors, folder markers, and `fetch_report`

`exists` returns False only for not-found; a 403 re-raises. A 0-byte `folder/` object (what a Databricks mkdir leaves on an S3 Volume) must be skipped by `list_json`. `fetch_report` downloads `history/<run>/` for the run named by `latest/pointer.json`, or an explicit run id.

In [ ]:
# a 403 should raise, not read as "missing"
if FORBIDDEN_S3:
    try:
        print("exists:", S3Store(FORBIDDEN_S3, client=store.client).exists("x"))
    except Exception as exc:
        print("raised:", type(exc).__name__, str(exc)[:200])
else:
    print("FORBIDDEN_S3 not set; skipped")

# a folder marker next to a real JSON object: list_json returns only the JSON
store.client.put_object(Bucket=store.bucket, Key=store._key("manifest/"), Body=b"")
store.put_json("manifest/a.json", meta)
print([key for key, _ in store.list_json("manifest/")])

In [ ]:
# fetch a report: from your real report store (read-only), or a small demo written under scratch
if REPORT_S3:
    reports = S3Store(REPORT_S3, client=store.client)
else:
    reports = S3Store(f"{ROOT}/report", client=store.client)
    html = f"<html><body><h1>{PAIR}</h1><pre>{json.dumps(identity, indent=1)}</pre></body></html>"
    reports.put_bytes(f"history/{RUN_TAG}/index.html", html.encode(), content_type="text/html")
    reports.put_json(f"history/{RUN_TAG}/summary.json", meta)
    reports.put_json("latest/pointer.json", {"run_id": RUN_TAG, "status": "DONE", "findings": 0})

latest = fetch_report(reports, WORKDIR / "report_latest")
print("latest run:", latest["run_id"])
for name, path in latest["files"].items():
    print(f"  {name:30} {path.stat().st_size:>8} bytes")

# an explicit run id that does not exist fails clearly
try:
    fetch_report(reports, WORKDIR / "report_nope", run_id="nope")
except FileNotFoundError as exc:
    print("FileNotFoundError:", exc)

## 8. Copy the profile into a static one (`refresh_profile`)

`refresh_profile(profile, target, path=...)` resolves fresh keys and writes only the `[target]` section of a credentials file, atomically, keeping every other section. Here the file is `WORKDIR/credentials`, seeded with one labelled dummy section to show it survives.

In [ ]:
# a scratch credentials file with an unrelated section that must survive (dummy text, not real keys)
cred_file = WORKDIR / "credentials"
cred_file.write_text("[other]\naws_access_key_id = DUMMY\n# keep me\n", encoding="utf-8")

# two refreshes: the second replaces only its own section
one = aws.refresh_profile(AWS_PROFILE, AWS_TARGET, path=cred_file)
two = aws.refresh_profile(AWS_PROFILE, AWS_TARGET, path=cred_file)
print(one)
print(two)
text = cred_file.read_text(encoding="utf-8")
print(f"[{AWS_TARGET}] sections:", text.count(f"[{AWS_TARGET}]"), "| [other] kept:", "# keep me" in text)
print("staging file left:", (cred_file.with_name("credentials.tmp")).exists())
print("mode:", oct(cred_file.stat().st_mode & 0o777), "(0o600 on POSIX; Windows ignores chmod)")
print(redact(text))

In [ ]:
# the static copy really works: point botocore at the scratch file and ask STS who it is
import botocore.session

bs = botocore.session.Session()
bs.set_config_variable("credentials_file", str(cred_file))
static = boto3.Session(botocore_session=bs, profile_name=AWS_TARGET, region_name=identity["region"] or "us-east-1")
print("method:", static.get_credentials().method)
print("arn   :", static.client("sts").get_caller_identity()["Arn"])
print("same as source:", static.client("sts").get_caller_identity()["Arn"] == identity["arn"])

In [ ]:
# the target may not be the source profile itself
try:
    aws.refresh_profile(AWS_PROFILE, AWS_PROFILE, path=cred_file)
except ValueError as exc:
    print("ValueError:", exc)

## 9. When to refresh next (`next_delay`)

`next_delay(expires_in, every)` refreshes every `every` seconds, or 5 minutes before expiry when that comes first, never sooner than 60 s. The first line uses your real expiry; the table below it is literal edge cases for reference.

In [ ]:
# your real credentials
print("expires_in:", two["expires_in"], "-> next refresh in", aws.next_delay(two["expires_in"], aws.DEFAULT_EVERY), "s")

# reference table (literal inputs)
pd.DataFrame([{"expires_in": e, "every": aws.DEFAULT_EVERY, "delay": aws.next_delay(e, aws.DEFAULT_EVERY)} for e in (None, 1800, 900, 200, -5)])

## 10. Keep it fresh (`watch`)

`watch(refresh, alive, every=, first=, tick=)` calls `refresh` on schedule while `alive()` is true and retries a failed refresh after 60 s. It runs for `WATCH_SECONDS`, refreshing the WORKDIR file for real.

In [ ]:
import time

# refresh into the scratch file; stop after WATCH_SECONDS
if WATCH_SECONDS:
    deadline = time.monotonic() + WATCH_SECONDS
    log = []

    def refresh():
        outcome = aws.refresh_profile(AWS_PROFILE, AWS_TARGET, path=cred_file)
        log.append((datetime.now().strftime("%H:%M:%S"), outcome["access_key"], outcome["expires_in"]))
        return outcome

    count = aws.watch(refresh, lambda: time.monotonic() < deadline, every=aws.MIN_DELAY, first=0, tick=2)
    print("refreshes:", count)
    for row in log:
        print(" ", row)

## 11. WinSCP

`winscp_path()` looks at `DTRACK_WINSCP`, the default install folders, then PATH, and fails with a clear message. `launch_winscp(site)` starts it on a saved site; it only runs when `LAUNCH_WINSCP = True`.

In [ ]:
# where WinSCP is, if anywhere
try:
    exe = aws.winscp_path()
    print("WinSCP:", exe)
except FileNotFoundError as exc:
    exe = None
    print("FileNotFoundError:", exc)

# optional: open the saved site (reads the static profile written by section 12)
if exe and LAUNCH_WINSCP:
    print(aws.launch_winscp(WINSCP_SITE, exe=exe).args)

## 12. Write the real static profile (DRY_RUN gated)

This is what `dtrack dbx s3 creds` does: write `[AWS_TARGET]` into the real shared credentials file. With `DRY_RUN = True` it only prints the plan.

In [ ]:
# the real file every AWS tool reads
target_file = aws.credentials_file()
if DRY_RUN:
    print(f"DRY_RUN: would write [{AWS_TARGET}] from profile {AWS_PROFILE!r} into {target_file}")
else:
    print(aws.refresh_profile(AWS_PROFILE, AWS_TARGET))

## Cleanup

Deletes every object under this run's S3 prefix and removes WORKDIR (which holds temporary keys) unless `KEEP = True`.

In [ ]:
# S3: everything under the run prefix
if KEEP:
    print("kept", ROOT, "and", WORKDIR)
else:
    prefix = store.location.prefix + "/"
    deleted = 0
    for page in store.client.get_paginator("list_objects_v2").paginate(Bucket=store.bucket, Prefix=prefix):
        keys = [{"Key": row["Key"]} for row in page.get("Contents") or []]
        if keys:
            store.client.delete_objects(Bucket=store.bucket, Delete={"Objects": keys})
            deleted += len(keys)
    print(f"deleted {deleted} object(s) under {ROOT}")
    shutil.rmtree(WORKDIR, ignore_errors=True)
    print("removed", WORKDIR)

## What to check

- `whoami` shows the account/ARN you expect and a positive `expires_in` (or None for static keys).
- Store contract: reads equal writes, `exists` is False for the missing key, `list_keys` returns relative names, raw keys all start with the run prefix.
- Pagination: more than one `list_objects_v2` call, `StartAfter` only on the first, `ContinuationToken` on the rest.
- `ready_unclaimed` goes 2 -> 0; heartbeat ages are 30.0 and 0.0; closing markers list sorted pairs.
- `list_json` skips the `manifest/` folder marker; `fetch_report` downloads the latest run and fails clearly for `nope`.
- The static profile in WORKDIR authenticates as the same ARN; only one `[target]` section; `[other]` kept.
- Cleanup deleted the run prefix.